## Projet ML P1 IA G5 : Segmentation Client

Groupe 5 :

* __Rokhaya Coumba Diouf__
* __Yohan Fredy Hounyouvi__
* __Khadidiatou Gueye__
* __Cheikh Ahmed Tidiane Ciss__

### Segmentation de la clientèle d'un grossiste

#### 0. Introduction

##### 0.1 Contexte métier
Un grossiste portugais en produits alimentaires approvisionne différents types de clients professionnels (B2B) : hôtels, restaurants, cafés, épiceries, supérettes, etc.

Aujourd'hui, il applique la **même stratégie** (logistique, offre commerciale, marketing) à tous ses clients, alors que leurs besoins et leurs volumes d'achat sont très différents.

Toute décision globale, par exemple réduire la fréquence des livraisons pour baisser les coûts, risque donc de **satisfaire certains clients et d'en mécontenter d'autres**. Le grossiste a besoin de mieux connaître sa clientèle pour adapter ses décisions à chaque type de client.

### 0.2 Problématique
> Peut-on identifier des groupes de clients homogènes à partir de leurs habitudes de dépenses, et en tirer des recommandations concrètes pour le grossiste ?

### 0.3 Objectifs
1. Explorer et comprendre le comportement d'achat des clients.
2. Construire une segmentation par **apprentissage non supervisé** (clustering).
3. Déterminer un nombre de segments pertinent et justifié.
4. Interpréter chaque segment (persona) et valider la segmentation.
5. Proposer des recommandations métier adaptées à chaque segment.

### 0.4 Description du jeu de données
**Source :** *Wholesale Customers Data Set*, UCI Machine Learning Repository.
Chaque ligne correspond à un client. Les variables de dépenses sont des **montants annuels** (en unités monétaires).

| Variable | Type | Description |
|---|---|---|
| `Channel` | Catégorielle | Canal de vente : 1 = Horeca (Hôtels/Restaurants/Cafés), 2 = Retail (commerce de détail) |
| `Region` | Catégorielle | Région : 1 = Lisbonne, 2 = Porto, 3 = Autre |
| `Fresh` | Numérique | Dépenses annuelles en produits frais |
| `Milk` | Numérique | Dépenses annuelles en produits laitiers |
| `Grocery` | Numérique | Dépenses annuelles en épicerie |
| `Frozen` | Numérique | Dépenses annuelles en surgelés |
| `Detergents_Paper` | Numérique | Dépenses annuelles en détergents et produits papier |
| `Delicassen` | Numérique | Dépenses annuelles en produits traiteur / épicerie fine |

### 0.5 Démarche générale
Le dataset ne contient **aucune variable cible** : il s'agit donc d'un problème d'**apprentissage non supervisé**.

**Choix méthodologique :** la segmentation sera construite uniquement à partir des **6 variables de dépenses**. Les variables `Channel` et `Region` seront mises de côté pour servir à **l'interprétation et à la validation** des segments. Si nos clusters retrouvent la distinction Horeca/Retail sans l'avoir vue pendant l'apprentissage, cela confirmera que la segmentation capte une structure réelle des données.

**Étapes du projet :**
1. Imports et chargement des données
2. Analyse exploratoire (EDA)
3. Prétraitement
4. Réduction de dimension (ACP)
5. Modélisation (K-Means, CAH, GMM)
6. Interprétation des segments
7. Evaluation et Validation
8. Affectation d'un nouveau client
9. Recommandations métier et conclusion

### 1. Le Machine Learning non supervisé

#### 1.1 Supervisé vs non supervisé
En Machine Learning, on distingue deux grandes familles d'apprentissage :

| | Apprentissage supervisé | Apprentissage non supervisé |
|---|---|---|
| **Données** | Variables explicatives **+ une variable cible** (étiquette) | Variables explicatives **uniquement**, sans étiquette |
| **Objectif** | Apprendre à **prédire** la cible | **Découvrir des structures cachées** dans les données |
| **Exemples de tâches** | Classification, régression | Clustering, réduction de dimension, détection d'anomalies |
| **Évaluation** | Comparaison prédiction / vraie valeur (accuracy, RMSE…) | Pas de « bonne réponse » : métriques internes (silhouette…) et interprétation métier |

#### 1.2 Pourquoi le non supervisé dans notre cas ?
Dans notre dataset, **aucune colonne n'indique à quel segment appartient chaque client** : les segments n'existent pas encore, c'est justement ce qu'on cherche à découvrir.
On ne peut donc pas entraîner un modèle supervisé. On utilise le **clustering**, qui regroupe automatiquement les clients selon la similarité de leurs dépenses.

#### 1.3 Le clustering
Le clustering consiste à partitionner les observations en groupes (**clusters**) tels que :
- les clients d'un **même cluster** soient **le plus similaires possible** (forte homogénéité intra-cluster) ;
- les clients de **clusters différents** soient **le plus différents possible** (forte séparation inter-clusters).

La similarité est généralement mesurée par une **distance** (souvent la distance euclidienne) entre les profils de dépenses. Cela implique que les variables soient **à la même échelle**, d'où l'importance du prétraitement (standardisation).

Principaux algorithmes envisagés dans ce projet :
- **K-Means** : affecte chaque client au centre de cluster le plus proche, en minimisant la variance intra-cluster. Le nombre de clusters *k* doit être choisi à l'avance.
- **Clustering hiérarchique (CAH)** : construit un arbre de regroupements successifs (dendrogramme), utile pour visualiser et choisir le nombre de groupes.
- **Modèles de mélange gaussien (GMM)** : approche probabiliste où chaque client a une probabilité d'appartenance à chaque cluster.

#### 1.4 Application à la segmentation client
Appliqué à notre problème, le clustering permettra de passer de **440 clients traités de la même façon** à **quelques segments** au comportement d'achat homogène, chacun pouvant faire l'objet d'une stratégie commerciale et logistique adaptée.

### Étape 3 : Chargement du dataset avec Pandas

#### 2. Chargement des données

##### 2.1 Import des bibliothèques
**Pourquoi :** on importe les bibliothèques utilisées dans cette partie :
- **Pandas** : manipulation de données tabulaires (DataFrame) ;
- **NumPy** : calcul numérique ;
- **Matplotlib** et **Seaborn** : visualisation.

On fixe aussi une **graine aléatoire** (`RANDOM_STATE`) pour garantir la **reproductibilité** des résultats, car certaines étapes du projet (K-Means, GMM) dépendent d'une initialisation aléatoire.

In [1]:
# --- Manipulation des données ---
import numpy as np
import pandas as pd

# --- Visualisation ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Divers ---
import warnings
warnings.filterwarnings("ignore")

# --- Paramètres globaux ---
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Style des graphiques
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)

# Affichage pandas : toutes les colonnes, 2 décimales
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

print("Bibliothèques importées avec succès.")

Bibliothèques importées avec succès.


### 2.2 Lecture du fichier CSV
**Pourquoi :** on charge le fichier dans un DataFrame Pandas, puis on affiche les premières lignes pour vérifier que la lecture s'est bien passée (bon séparateur, bons noms de colonnes, valeurs cohérentes).

In [2]:
# Chargement du dataset
df = pd.read_csv("data/Wholesale customers data.csv")

print("Dataset chargé avec succès.")
df.head()

Dataset chargé avec succès.


,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
0,2,3,12669,9656,7561,214,2674,1338
1,2,3,7057,9810,9568,1762,3293,1776
2,2,3,6353,8808,7684,2405,3516,7844
3,1,3,13265,1196,4221,6404,507,1788
4,2,3,22615,5410,7198,3915,1777,5185


In [3]:
# Aperçu des dernières lignes et d'un échantillon aléatoire
display(df.tail())
display(df.sample(5, random_state=RANDOM_STATE))

,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
435,1,3,29703,12051,16027,13135,182,2204
436,1,3,39228,1431,764,4510,93,2346
437,2,3,14531,15488,30243,437,14841,1867
438,1,3,10290,1981,2232,1038,168,2125
439,1,3,2787,1698,2510,65,477,52


,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
265,1,1,5909,23527,13699,10155,830,3636
78,1,3,10766,1175,2067,2096,301,167
347,2,3,27380,7184,12311,2809,4621,1022
255,1,1,31614,489,1495,3242,111,615
327,1,2,542,899,1664,414,88,522


### 2.3 Définition des groupes de variables
**Pourquoi :** on sépare dès maintenant les variables en deux groupes, qu'on réutilisera dans tout le notebook :
- les **variables catégorielles** (`Channel`, `Region`), qui serviront à l'interprétation et à la validation ;
- les **variables de dépenses**, qui serviront à construire la segmentation.

On crée aussi des dictionnaires de correspondance pour remplacer les codes numériques par des libellés lisibles dans les graphiques.

In [4]:
# Groupes de variables
CAT_COLS = ["Channel", "Region"]
SPEND_COLS = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]

# Correspondance codes -> libellés
channel_map = {1: "Horeca", 2: "Retail"}
region_map = {1: "Lisbonne", 2: "Porto", 3: "Autre"}

print("Variables catégorielles :", CAT_COLS)
print("Variables de dépenses   :", SPEND_COLS)

Variables catégorielles : ['Channel', 'Region']
Variables de dépenses   : ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']


**Observation :** le fichier est correctement chargé. Les 8 colonnes attendues sont présentes, avec des noms cohérents avec la description du dataset. Les variables de dépenses prennent des valeurs très variables d'un client à l'autre (de quelques unités à plusieurs dizaines de milliers), ce que nous analyserons en détail dans l'exploration.